# Exploratory Data Analysis - Fraud Detection

This notebook demonstrates how to load the generated synthetic transactions data and perform basic exploratory data analysis (EDA) to understand the dataset structure and features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Data
First, we load the raw transactions dataset generated by the pipeline.

In [ ]:
import os
data_path = os.path.join('..', 'data', 'raw', 'transactions.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    print(f"Raw data file not found at {data_path}. Please run main.py or src/data_generator.py first.")
    # Generate dummy data for illustration if notebook is run in isolation without file
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=1000, n_features=5, weights=[0.98, 0.02], random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])
    df['is_fraud'] = y
    df['amount'] = np.random.lognormal(mean=3.5, sigma=1.0, size=1000)
    df['distance_from_home'] = np.random.lognormal(mean=1.5, sigma=1.2, size=1000)
    print("Using generated fallback dataset for visualization.")

df.head()

## 2. Class Imbalance Check
Fraud detection problems are notoriously imbalanced. Let's check the distribution of the class label `is_fraud`.

In [ ]:
class_counts = df['is_fraud'].value_counts()
class_pct = df['is_fraud'].value_counts(normalize=True) * 100
print("Class Counts:\n", class_counts)
print("\nClass Percentages:\n", class_pct)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='is_fraud', hue='is_fraud', palette='Set1', legend=False)
plt.title('Distribution of Transactions (0: Legit, 1: Fraud)')
plt.xlabel('Is Fraud?')
plt.ylabel('Count')
plt.show()

## 3. Exploratory Plots
Let's look at key features like transaction `amount` and `distance_from_home` relative to fraud status.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount vs Fraud
sns.boxplot(ax=axes[0], data=df, x='is_fraud', y='amount', hue='is_fraud', palette='Set2', legend=False)
axes[0].set_title('Transaction Amount by Class')
axes[0].set_yscale('log') # Log scale because amount has log-normal distribution
axes[0].set_xlabel('Is Fraud?')
axes[0].set_ylabel('Amount (Log Scale)')

# Distance vs Fraud
sns.boxplot(ax=axes[1], data=df, x='is_fraud', y='distance_from_home', hue='is_fraud', palette='Set2', legend=False)
axes[1].set_title('Distance from Home by Class')
axes[1].set_yscale('log')
axes[1].set_xlabel('Is Fraud?')
axes[1].set_ylabel('Distance (Log Scale)')

plt.tight_layout()
plt.show()

## 4. Fraud vs Hour of Day

In [ ]:
if 'hour' in df.columns:
    plt.figure(figsize=(12, 5))
    sns.kdeplot(data=df, x='hour', hue='is_fraud', common_norm=False, fill=True, palette='Set1')
    plt.title('Transaction Density over Hours of the Day')
    plt.xlabel('Hour of Day')
    plt.ylabel('Density')
    plt.xlim(0, 23)
    plt.show()
else:
    print("Hour column not present in the current dataset views.")